# GPT 모델 직접 구현 - 실습 코드 1: GPT 모델 구현

- Tutorial ID: `expand-gpt-from-scratch`
- Tutorial: GPT 모델 직접 구현
- Section ID: `expand-gpt-from-scratch-code-1`
- Section: 실습 코드 1: GPT 모델 구현


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: GPT 모델 구현
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) 미래 토큰을 -inf로 막은 뒤 softmax 확률이 0이 되는지 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class GPTBlock(nn.Module):
        def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
                        d_model, num_heads, dropout=dropout, batch_first=True
        )
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

        def forward(self, x, mask=None):
        h = self.ln1(x)
                attn_out, _ = self.attn(h, h, h, attn_mask=mask, is_causal=True)
                x = x + attn_out
                x = x + self.ffn(self.ln2(x))
        return x

class GPT(nn.Module):
        def __init__(self, vocab_size, d_model=768, num_heads=12,
                 d_ff=3072, num_layers=12, max_seq_len=1024):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.blocks = nn.ModuleList([
            GPTBlock(d_model, num_heads, d_ff) for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        # Weight tying
        self.head.weight = self.tok_emb.weight

        def forward(self, idx):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
                x = self.tok_emb(idx) + self.pos_emb(pos)
                for block in self.blocks:
                        x = block(x)
                x = self.ln_f(x)
                logits = self.head(x)
        return logits

    @torch.no_grad()
        def generate(self, idx, max_new_tokens, temperature=1.0):
                for _ in range(max_new_tokens):
            idx_cond = idx[:, -1024:]
                        logits = self(idx_cond)[:, -1, :] / temperature
                        probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

# GPT-2 Small 규모
model = GPT(vocab_size=50257, d_model=768, num_heads=12, d_ff=3072, num_layers=12)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")  # ~124M